In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import cv2
from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from tqdm import tqdm


c:\Users\nviquang\Documents\University\ThirdYear\Ky2\TTCS\Repo\SourceCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = "./data_ready_for_kaggle"
IMG_DIR = os.path.join(DATA_DIR, "images_resized")

class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=50):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        image_file = self.df.iloc[index]['image_file']
        caption = self.df.iloc[index]['caption']
        
        image_path = os.path.join(self.img_dir, image_file)
        
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        encoding = self.processor(
            images=image,
            text=caption,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        labels = encoding["input_ids"].clone()
        pad_token_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_token_id] = -100
        encoding["labels"] = labels
        
        return encoding 


In [3]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {device}")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

for param in model.vision_model.parameters():
    param.requires_grad = False

train_dataset = UITViICDataset(os.path.join(DATA_DIR, "mini_train.csv"), IMG_DIR, processor)
dev_dataset = UITViICDataset(os.path.join(DATA_DIR, "cleaned_dev.csv"), IMG_DIR, processor)

BATCH_SIZE = 4 
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
dev_dataloader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


Đang chạy trên: cuda


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 18437.27it/s]
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:

LEARNING_RATE = 5e-5  # Thử nghiệm: 1e-4, 5e-5, 2e-5
EPOCHS = 3           
ACCUMULATION_STEPS = 16 

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
scaler = GradScaler() 



C:\Users\nviquang\AppData\Local\Temp\ipykernel_9872\2281439034.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:

print(f"\n--- Bắt đầu Tuning với LR={LEARNING_RATE} ---")
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    running_loss, train_loss_total = 0.0, 0.0
    optimizer.zero_grad()
    
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for step, batch in progress_bar:
        pixel_values = batch['pixel_values'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch["labels"].to(device)

        with autocast():
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss / ACCUMULATION_STEPS
        
        scaler.scale(loss).backward()
        
        running_loss += loss.item() * ACCUMULATION_STEPS
        train_loss_total += loss.item() * ACCUMULATION_STEPS

        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        if (step + 1) % 50 == 0:
            progress_bar.set_postfix({'loss': f"{(running_loss / 50):.4f}"})
            running_loss = 0.0
            
    model.eval()
    val_loss_total = 0.0
    with torch.no_grad():
        for batch in tqdm(dev_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            with autocast():
                outputs = model(
                    pixel_values=batch['pixel_values'].to(device),
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                    labels=batch["labels"].to(device)
                )
            val_loss_total += outputs.loss.item()
            
    avg_train_loss = train_loss_total / len(train_dataloader)
    avg_val_loss = val_loss_total / len(dev_dataloader)
    
    print(f"-> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


--- Bắt đầu Tuning với LR=5e-05 ---
